In [1]:
import pandas as pd
import os

# Rutas a los archivos
data_path = '../data/raw/'
files = ['2015.csv', '2016.csv', '2017.csv', '2018.csv', '2019.csv']

# Cargar los datasets en un diccionario para explorarlos fácilmente
datasets = {}
for file in files:
    year = file.split('.')[0]
    datasets[year] = pd.read_csv(os.path.join(data_path, file))
    print(f"Dataset {year} cargado con dimensiones: {datasets[year].shape}")

Dataset 2015 cargado con dimensiones: (158, 12)
Dataset 2016 cargado con dimensiones: (157, 13)
Dataset 2017 cargado con dimensiones: (155, 12)
Dataset 2018 cargado con dimensiones: (156, 9)
Dataset 2019 cargado con dimensiones: (156, 9)


In [2]:
# 1. Comparar los nombres de las columnas (Esquemas)
print("COMPRABACIÓN DE ESQUEMAS Y COLUMNAS\n" + "="*40)
for year, df in datasets.items():
    print(f"\n--- Columnas del año {year} ---")
    print(df.columns.tolist())

COMPRABACIÓN DE ESQUEMAS Y COLUMNAS

--- Columnas del año 2015 ---
['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Standard Error', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']

--- Columnas del año 2016 ---
['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Lower Confidence Interval', 'Upper Confidence Interval', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']

--- Columnas del año 2017 ---
['Country', 'Happiness.Rank', 'Happiness.Score', 'Whisker.high', 'Whisker.low', 'Economy..GDP.per.Capita.', 'Family', 'Health..Life.Expectancy.', 'Freedom', 'Generosity', 'Trust..Government.Corruption.', 'Dystopia.Residual']

--- Columnas del año 2018 ---
['Overall rank', 'Country or region', 'Score', 'GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life

In [3]:
# 2. Revisar tipos de datos y valores nulos
print("\nTIPOS DE DATOS Y VALORES NULOS\n" + "="*40)
for year, df in datasets.items():
    print(f"\n--- Resumen del año {year} ---")
    # Mostrar nulos
    nulos = df.isnull().sum()
    print("Valores nulos encontrados:")
    print(nulos[nulos > 0] if nulos.sum() > 0 else "Ningún valor nulo")


TIPOS DE DATOS Y VALORES NULOS

--- Resumen del año 2015 ---
Valores nulos encontrados:
Ningún valor nulo

--- Resumen del año 2016 ---
Valores nulos encontrados:
Ningún valor nulo

--- Resumen del año 2017 ---
Valores nulos encontrados:
Ningún valor nulo

--- Resumen del año 2018 ---
Valores nulos encontrados:
Perceptions of corruption    1
dtype: int64

--- Resumen del año 2019 ---
Valores nulos encontrados:
Ningún valor nulo


In [4]:
# Paso 2: Limpieza y Armonización (Data Cleaning and Harmonization)

# 1. Definir un diccionario de mapeo para estandarizar las columnas de cada año
column_mapping = {
    '2015': {
        'Country': 'country',
        'Happiness Score': 'actual_happiness_score',
        'Economy (GDP per Capita)': 'gdp',
        'Family': 'family',
        'Health (Life Expectancy)': 'health',
        'Freedom': 'freedom',
        'Generosity': 'generosity',
        'Trust (Government Corruption)': 'corruption'
    },
    '2016': {
        'Country': 'country',
        'Happiness Score': 'actual_happiness_score',
        'Economy (GDP per Capita)': 'gdp',
        'Family': 'family',
        'Health (Life Expectancy)': 'health',
        'Freedom': 'freedom',
        'Generosity': 'generosity',
        'Trust (Government Corruption)': 'corruption'
    },
    '2017': {
        'Country': 'country',
        'Happiness.Score': 'actual_happiness_score',
        'Economy..GDP.per.Capita.': 'gdp',
        'Family': 'family',
        'Health..Life.Expectancy.': 'health',
        'Freedom': 'freedom',
        'Generosity': 'generosity',
        'Trust..Government.Corruption.': 'corruption'
    },
    '2018': {
        'Country or region': 'country',
        'Score': 'actual_happiness_score',
        'GDP per capita': 'gdp',
        'Social support': 'family',
        'Healthy life expectancy': 'health',
        'Freedom to make life choices': 'freedom',
        'Generosity': 'generosity',
        'Perceptions of corruption': 'corruption'
    },
    '2019': {
        'Country or region': 'country',
        'Score': 'actual_happiness_score',
        'GDP per capita': 'gdp',
        'Social support': 'family',
        'Healthy life expectancy': 'health',
        'Freedom to make life choices': 'freedom',
        'Generosity': 'generosity',
        'Perceptions of corruption': 'corruption'
    }
}

processed_dfs = []

# 2. Procesar cada dataset
for year, df in datasets.items():
    # Crear una copia para manipular
    temp_df = df.copy()
    
    # Renombrar columnas
    temp_df = temp_df.rename(columns=column_mapping[year])
    
    # Conservar únicamente las columnas requeridas para el análisis
    columns_to_keep = list(column_mapping[year].values())
    temp_df = temp_df[columns_to_keep]
    
    # Agregar la columna 'year' correspondiente a cada archivo
    temp_df['year'] = int(year)
    
    # 3. Manejo de valores nulos (Rellenar el nulo de corrupción de 2018 con 0)
    if year == '2018':
        temp_df['corruption'] = temp_df['corruption'].fillna(0)
        
    processed_dfs.append(temp_df)

# 4. Fusionar todos los años en un solo DataFrame
unified_df = pd.concat(processed_dfs, ignore_index=True)

print("¡Datasets unificados con éxito!")
print(f"Dimensiones del nuevo dataset: {unified_df.shape}\n")
print("Información del dataset:")
print(unified_df.info())

¡Datasets unificados con éxito!
Dimensiones del nuevo dataset: (782, 9)

Información del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 782 non-null    str    
 1   actual_happiness_score  782 non-null    float64
 2   gdp                     782 non-null    float64
 3   family                  782 non-null    float64
 4   health                  782 non-null    float64
 5   freedom                 782 non-null    float64
 6   generosity              782 non-null    float64
 7   corruption              782 non-null    float64
 8   year                    782 non-null    int64  
dtypes: float64(7), int64(1), str(1)
memory usage: 55.1 KB
None


In [5]:
# Guardar el dataset unificado procesado
import os
os.makedirs('../data/processed', exist_ok=True)
unified_df.to_csv('../data/processed/unified_happiness.csv', index=False)
print("¡Dataset guardado en data/processed/unified_happiness.csv!")

¡Dataset guardado en data/processed/unified_happiness.csv!
